# Module 00 · Unit 01
# Proof Techniques for Security Reasoning

| | |
|---|---|
| **Estimated time** | 50–60 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Foundations of formal security reasoning \[AC\] \[AG\] |
| **Refresher** | No prerequisites — this is the starting point |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why mathematical proof is essential — not optional — in security engineering
2. Construct a **direct proof** of a simple security claim
3. Apply **proof by contrapositive** and **proof by contradiction**
4. Use a **counterexample** to decisively refute a security claim
5. Define a system **invariant** and reason about whether it is preserved
6. Use Python to search for counterexamples and monitor invariants programmatically


## 🔣 Symbol Primer

New to discrete math notation? This table covers every symbol used in this notebook.
Keep it open in a split screen while you read. After a few notebooks the symbols will
feel automatic — but there is no shame in looking them up every single time until then.

| Symbol | Read it as | What it means |
|---|---|---|
| $\forall$ | "for all" | The property holds for **every** element of the set |
| $\exists$ | "there exists" | There is **at least one** element for which the property holds |
| $\in$ | "is an element of" | The thing on the left belongs to the set on the right: $3 \in \mathbb{Z}$ |
| $\notin$ | "is not an element of" | The thing on the left is **not** in the set: $0.5 \notin \mathbb{Z}$ |
| $\subseteq$ | "is a subset of" | Every element of the left set also lives in the right set |
| $\mathbb{N}$ | "the naturals" | The counting numbers: $0, 1, 2, 3, \ldots$ |
| $\mathbb{Z}$ | "the integers" | Whole numbers including negatives: $\ldots, -2, -1, 0, 1, 2, \ldots$ |
| $\mathbb{R}$ | "the reals" | All numbers on the continuous number line |
| $\neg$ | "not" | Flips truth to false and false to truth: $\neg\,\mathtt{true} = \mathtt{false}$ |
| $\wedge$ | "and" | True **only when both** sides are true |
| $\vee$ | "or" | True when **at least one** side is true |
| $\Rightarrow$ | "implies" / "if...then" | If the left is true, the right **must** be true |
| $\Longleftrightarrow$ | "if and only if" / "iff" | Both sides are true together or false together |
| $\square$ | "end of proof" / "QED" | Marks that the argument is complete |
| $\Rightarrow\!\Leftarrow$ | "contradiction" | We have derived an impossibility — the assumption must be wrong |

> **Tip.** When you see a symbol you don't recognise in running text, ask: is it about
> *membership* ($\in$, $\subseteq$), *logic* ($\neg$, $\wedge$, $\vee$, $\Rightarrow$,
> $\Longleftrightarrow$), or *quantification* ($\forall$, $\exists$)?
> Those three categories cover the vast majority of discrete math notation.

---


## 1 · Why Formal Proof Matters in Security

Security engineering is full of claims:

> *"This input validation rule blocks all injection attempts."*  
> *"A user cannot escalate privileges without administrator approval."*  
> *"After three failed login attempts, the account is always locked."*

Each of these is a **universal claim** — it asserts that something holds for *all* inputs,
*all* users, *all* sequences of events. Universal claims carry a devastating asymmetry:

$$\text{To prove a universal claim: you must reason about ALL cases.}$$

$$\text{To disprove a universal claim: you need exactly ONE counterexample.}$$

This asymmetry is the core of adversarial thinking. An attacker is not trying to prove
your system is insecure in general. They are looking for **one input**, **one state**,
**one sequence of events** where your claimed guarantee fails. A formal proof, when it
exists, closes that door completely.

### What Is a Proof?

A **proof** is a finite sequence of logical steps that establishes the truth of a
statement beyond doubt, given a set of agreed-upon axioms and inference rules. Every
step must follow from what came before by a valid rule of inference.

Proofs are not only for mathematicians. Every time a compiler verifies type safety,
every time a protocol is checked against a formal specification, every time a
policy engine evaluates an access control decision — these computations embody formal
reasoning. In this course you will learn to write claims precisely enough that they
can be proved or refuted, and to apply the main proof techniques to real security
properties.


## 2 · Propositions and Precise Claims

A **proposition** is a statement that is either true or false — not both, not neither.

| Statement | Proposition? | Why |
|---|---|---|
| "The sky is blue." | Yes | Has a definite truth value |
| "Is this input safe?" | No | A question, not a claim |
| "Please validate all inputs." | No | A command |
| "For all $n \in \mathbb{N}$, $2n$ is even." | Yes | Universally quantified |
| "There exists an input $x$ such that $f(x) = 0$." | Yes | Existentially quantified |

### Universal vs. Existential Claims

The most important distinction in security reasoning is between claims that say
*"this holds for everything"* and claims that say *"there is at least one case."*

**Universal claim** — uses $\forall$ *(for all)*:

$$\forall x \in \mathcal{I},\quad \text{validate}(x) = \mathtt{true} \Rightarrow \text{safe}(x) = \mathtt{true}$$

Read aloud: *"For all inputs $x$ in $\mathcal{I}$ *(our input set)*, if validate returns
true then safe also returns true."*

**Existential claim** — uses $\exists$ *(there exists)*:

$$\exists\, x \in \mathcal{I},\quad \text{validate}(x) = \mathtt{true} \;\wedge\; \text{safe}(x) = \mathtt{false}$$

Read aloud: *"There exists some input $x$ such that validate returns true **and**
($\wedge$, *(and)*) safe returns false."*

Notice that the existential claim is the **negation** of the universal claim. If you
can produce even one such $x$, the universal claim is false. This is why counterexample
search is so powerful — a single case breaks the whole guarantee.

> **Symbol check.** In the universal claim: $\mathcal{I}$ is just a name for
> the set of all possible inputs; the curly $\mathcal{I}$ is conventional for sets
> with a special role. The symbol $\in$ *(is an element of)* says $x$ is drawn from
> that set. $\mathbb{N}$ in the table above refers to the natural numbers
> $\{0, 1, 2, 3, \ldots\}$; $\mathbb{Z}$ refers to all integers
> $\{\ldots, -2, -1, 0, 1, 2, \ldots\}$.

We will study $\forall$ and $\exists$ formally in Module 02. For now, train yourself to
notice them in every security claim you read.


## 3 · Direct Proof

A **direct proof** establishes $P \Rightarrow Q$ *(if P then Q)* by assuming $P$ is
true and deriving $Q$ through a chain of valid steps.

**Structure:**
1. Assume $P$.
2. Apply definitions, previously proved theorems, and algebraic manipulations.
3. Arrive at $Q$. Conclude $P \Rightarrow Q$. Mark with $\square$ *(end of proof)*.

---

### Example — Transitivity of Role Permissions

**Setup.** Let $\text{perms}(R)$ denote the set of permissions held by role $R$.
We say role $A$ *dominates* role $B$ when $\text{perms}(B) \subseteq \text{perms}(A)$
— $\subseteq$ *(is a subset of)* means everything $B$ can do, $A$ can also do.

**Claim.** If $A$ dominates $B$ and $B$ dominates $C$, then $A$ dominates $C$.

Formally:
$$\text{perms}(B) \subseteq \text{perms}(A) \;\wedge\; \text{perms}(C) \subseteq \text{perms}(B) \;\Rightarrow\; \text{perms}(C) \subseteq \text{perms}(A)$$

**Proof.**

Assume $\text{perms}(B) \subseteq \text{perms}(A)$ and $\text{perms}(C) \subseteq \text{perms}(B)$.

Let $p$ be any element of $\text{perms}(C)$. We must show $p \in \text{perms}(A)$.

Since $\text{perms}(C) \subseteq \text{perms}(B)$, we have $p \in \text{perms}(B)$.

Since $\text{perms}(B) \subseteq \text{perms}(A)$, we have $p \in \text{perms}(A)$.

Since $p$ was an arbitrary element of $\text{perms}(C)$:

$$\text{perms}(C) \subseteq \text{perms}(A) \qquad \square$$

---

**Why this matters.** This is the theorem behind role hierarchy transitivity in RBAC.
If the hierarchy is defined correctly, privilege inheritance composes without surprises
— a property that RBAC security models depend on. We prove a generalized version using
set theory in Module 03.


## 4 · Proof by Contrapositive

The statement $P \Rightarrow Q$ is logically equivalent to its **contrapositive**
$\neg Q \Rightarrow \neg P$, where $\neg$ *(not)* flips the truth value of a statement:

$$\bigl(P \Rightarrow Q\bigr) \;\Longleftrightarrow\; \bigl(\neg Q \Rightarrow \neg P\bigr)$$

The double-headed arrow $\Longleftrightarrow$ *(if and only if)* means both directions
hold simultaneously — the two statements are completely interchangeable. This is not
a trick; it is a logical identity that we will prove with a truth table in Module 01.

When a direct proof is awkward, proving the contrapositive is often cleaner because
you start from a concrete failure condition ($\neg Q$) rather than an abstract premise.

**Structure:**
1. Assume $\neg Q$ *(the conclusion fails)*.
2. Derive $\neg P$ *(the premise must also fail)*.
3. Conclude: $P \Rightarrow Q$. $\square$

---

### Example — Certificate Validity and Connection Security

**Claim.** If a TLS connection $c$ is secure, then the certificate chain of $c$ is valid.

$$\text{secure}(c) \Rightarrow \text{valid\_chain}(c)$$

**Proof by contrapositive.** We prove
$\neg\,\text{valid\_chain}(c) \Rightarrow \neg\,\text{secure}(c)$.

Assume the certificate chain is not valid — at least one certificate was not signed by
a trusted authority, has expired, or has been revoked. In any of these cases, the
client cannot establish that it is communicating with the intended party, so the
connection cannot be considered secure. Therefore $\neg\,\text{secure}(c)$. $\square$

---

**The practitioner insight.** The contrapositive direction is how security monitoring
works in practice: you observe a symptom ($\neg Q$) and infer a structural condition
($\neg P$). Seeing an invalid certificate *is* evidence of an insecure connection —
you don't need to know *why* it is invalid to know the connection should not be trusted.


## 5 · Proof by Contradiction

In a **proof by contradiction** (*reductio ad absurdum*), you assume the negation of
your claim and show that this leads to a logical impossibility.

**Structure:**
1. Assume $\neg P$ — the claim you want to prove is false.
2. Derive consequences until you obtain both $Q$ and $\neg Q$ for some statement $Q$.
3. Two contradictory statements cannot both be true. Mark the absurdity with
   $\Rightarrow\!\Leftarrow$ *(contradiction — we've derived an impossibility)*.
4. Conclude: the assumption $\neg P$ must be wrong, so $P$ is true. $\square$

---

### Example — Uniqueness Under Injectivity

**Setup.** An encryption scheme $E$ is **injective** if distinct keys applied to the
same plaintext always produce distinct ciphertexts:

$$k_1 \neq k_2 \Rightarrow E_{k_1}(m) \neq E_{k_2}(m)$$

**Claim.** If an injective scheme is consistent (i.e., $D_k(E_k(m)) = m$ for all
$k, m$), and two keys $k_1, k_2$ both decrypt the same ciphertext $c$ to the same
plaintext $m$, then $k_1 = k_2$.

**Proof by contradiction.** Assume $k_1 \neq k_2$.

By injectivity: $E_{k_1}(m) \neq E_{k_2}(m)$. $\quad (*)$

By hypothesis, $D_{k_1}(c) = m$ and $D_{k_2}(c) = m$.
By consistency, this means $E_{k_1}(m) = c$ and $E_{k_2}(m) = c$.

Therefore $E_{k_1}(m) = E_{k_2}(m)$. $\quad (**)$

Statements $(*)$ and $(**)$ cannot both be true at once. $\Rightarrow\!\Leftarrow$

The assumption $k_1 \neq k_2$ is false. Therefore $k_1 = k_2$. $\square$

---

We will apply this technique repeatedly in Module 08 when proving properties of modular
arithmetic that underpin RSA and elliptic curve cryptography.


## 6 · Counterexamples — The Security Practitioner's Sharpest Tool

To **disprove** a universal claim $\forall x \in S,\ P(x)$, it suffices to exhibit
a single $x_0 \in S$ such that $P(x_0)$ is false. This $x_0$ is a **counterexample**.

$$\text{Counterexample to } \forall x \in S,\ P(x): \quad \text{find } x_0 \in S \text{ such that } \neg P(x_0).$$

**Security reframings of this idea:**

- A penetration tester is a professional counterexample finder.
- A CVE is a documented counterexample to a claimed security property.
- Fuzzing is automated counterexample search over an input space.
- A unit test is a manual counterexample check for a specific case.

### Structure of a Counterexample Refutation

**Claimed property:** $\forall x \in \mathcal{I},\ P(x)$

**Refutation:**

> Let $x_0 = [\text{specific value}]$. Then $\neg P(x_0)$ because $[\text{reason}]$.  
> Therefore the claim $\forall x \in \mathcal{I},\ P(x)$ is **false**. $\square$

Note that this is itself a proof — a proof of the negation.

---

### Example — Input Length Is Not a Security Guarantee

**Claimed property:** "Any input of length $\leq 256$ bytes is safe to process."

**Counterexample.** Let $x_0 = $ `'` (a single quote, 1 byte). Then $|x_0| = 1 \leq 256$,
so the length condition is satisfied. However, this input terminates a SQL string
literal and allows injection of arbitrary SQL. Therefore $\neg\,\text{safe}(x_0)$
holds despite $|x_0| \leq 256$. The claim is false. $\square$

---

This is one of the most important lessons in security: **syntactic properties** (length,
character set, encoding) do not imply **semantic safety**. Writing claims formally makes
the gap visible before an attacker finds it.


## 7 · Invariants

An **invariant** is a property of a system that holds true at every point during
execution — before, during, and after any operation.

Formally, let $\mathcal{S}$ be the set of all possible system states, $s_0 \in \mathcal{S}$
the initial state, and $\text{step} : \mathcal{S} \to \mathcal{S}$ a state transition.
A predicate $I : \mathcal{S} \to \{\mathtt{true}, \mathtt{false}\}$ is an **invariant** if:

$$I(s_0) = \mathtt{true} \qquad \text{(holds initially)}$$

$$\forall s \in \mathcal{S},\quad I(s) = \mathtt{true} \Rightarrow I(\text{step}(s)) = \mathtt{true} \qquad \text{(preserved by every transition)}$$

Read the second line aloud: *"For all states $s$, if the invariant holds now, then it
still holds after one step."* Together with the first line, this guarantees the
invariant holds in every reachable state — a fact we will prove formally using
induction in the Module 00 extension.

### Security Invariants in Practice

| System | Invariant |
|---|---|
| Access control (RBAC) | A user never holds permissions beyond their assigned role |
| Memory safety | Every pointer dereference falls within a valid allocation |
| TLS handshake | A session key is never used before mutual authentication completes |
| Agentic AI | An agent never invokes a tool it was not explicitly granted access to |
| Blockchain | The sum of all balances never exceeds the total supply |

A **security vulnerability** is often exactly an invariant violation: the system reached
a state that the designer believed was unreachable. The attacker's job is to find a
sequence of valid-looking transitions that breaks the invariant.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AC\] \[AG\]

The five techniques in this notebook are not warm-up exercises. They are the formal
machinery behind every rigorous security argument you will make in this course — and
in the research literature.

| Technique | Where it reappears in this course |
|---|---|
| **Direct proof** | Module 03: proving a function is a bijection (used in crypto key spaces) |
| **Proof by contrapositive** | Module 05: if no path exists in the attack graph, lateral movement is blocked |
| **Proof by contradiction** | Module 08: proving uniqueness of modular inverses for RSA |
| **Counterexample** | Module 01: one truth-table row that violates an access control policy |
| **Invariant** | Module 09: proving a protocol state machine never enters an illegal state |

**The practitioner test.** When you encounter any security claim — in a paper, a CVE,
a vendor whitepaper — ask two questions:

1. Is this a **universal claim** ($\forall$)? If so, what would a counterexample look like?
2. Is there an **invariant** here? Is there a proof that every transition ($\Rightarrow$)
   preserves it?

If neither question has a clean, formal answer, the claim is not yet a rigorous
security guarantee.

---


## Python: Visualization & Verification

The three sections below use Python to make the abstract machinery above concrete.

**1 · Counterexample Search** — automate the search for counterexamples to a
proposed property and visualize which cases satisfy it and which do not.

**2 · Invariant Monitoring** — simulate a privilege-level system, define a safety
invariant, and compare a correct run against a vulnerable run where a bug allows the
invariant to be broken.

**3 · Proof Strategy Visualizer** — display the logical flow of all three proof
strategies side by side so their structural differences are clear at a glance.


In [ ]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)

print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sympy as sp
import networkx as nx

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

# Course colour palette — same as Threat Surfaces
TS_BLUE  = '#1e64b4'   # Primary — main graph elements, logic nodes
TS_AMBER = '#c87814'   # Accent — attack paths, highlighted edges
TS_GREEN = '#1e8c50'   # Security bridge, safe/valid states
TS_GRAY  = '#64646e'   # Axes, secondary text, inactive nodes
TS_RED   = '#b41e1e'   # Vulnerabilities, contradictions, violations
TS_LIGHT = '#f5f7fa'   # Background tints, annotation boxes

print('Setup complete.')


### 1 · Automated Counterexample Search

Consider the following claim about prime numbers:

**Claimed property:** *"Every odd integer greater than 1 is prime."*

Formally: $\forall n \in \mathbb{Z},\ n > 1 \wedge n \text{ is odd} \Rightarrow n \text{ is prime}$

This is false — but let's let the computer find the counterexamples for us. The code
below defines `is_odd` and `is_prime`, scans all odd integers in $[3, 50]$, and plots
each one as either satisfying the claimed property (green circle) or violating it
(red ×). Violated cases are the counterexamples.

This is a toy example, but the pattern — *write the property as a predicate, scan the
domain, flag violations* — is exactly what fuzzers and property-based testing tools do
at scale for real security properties.


In [ ]:
# ── 1 · Counterexample Search ─────────────────────────────────────────────────

def is_odd(n):
    return n % 2 != 0

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

# Scan all odd integers in [3, 50]
odd_results = [(n, is_prime(n)) for n in range(3, 51) if is_odd(n)]
counterexamples = [n for n, prime in odd_results if not prime]

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 3.5))

for n, prime in odd_results:
    color  = TS_GREEN if prime else TS_RED
    marker = 'o'       if prime else 'X'
    ax.scatter(n, 1, color=color, marker=marker, s=130, zorder=3, linewidths=1.4,
               edgecolors='white')

# Annotate the first four counterexamples
for cx in counterexamples[:4]:
    factor = next(i for i in range(3, cx) if cx % i == 0)
    ax.annotate(
        f'{cx} = {factor}×{cx // factor}',
        xy=(cx, 1), xytext=(cx, 1.22),
        ha='center', fontsize=9, color=TS_RED, fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=TS_RED, lw=1.2)
    )

ax.set_xlim(1, 52)
ax.set_ylim(0.6, 1.55)
ax.set_yticks([])
ax.set_xlabel('$n$ — odd integers, $3 \leq n \leq 50$')
ax.set_title(
    'Claim: "Every odd integer $n > 1$ is prime"\n'
    'Searching for counterexamples in $[3, 50]$...',
    pad=10
)

handles = [
    mpatches.Patch(color=TS_GREEN, label='Odd and prime — claim holds for this $n$'),
    mpatches.Patch(color=TS_RED,   label='Odd but composite — counterexample!'),
]
ax.legend(handles=handles, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print(f"Counterexamples found: {counterexamples}")
print(f"The claim is FALSE. Smallest counterexample: n = {counterexamples[0]}")
print(f"  {counterexamples[0]} is odd, but {counterexamples[0]} = "
      f"{next(i for i in range(3, counterexamples[0]) if counterexamples[0] % i == 0)}"
      f" × {counterexamples[0] // next(i for i in range(3, counterexamples[0]) if counterexamples[0] % i == 0)}")


**What do you see?**

- The green circles are odd primes: 3, 5, 7, 11, 13, ... — the claim holds for these.
- The red × marks are odd composites: 9, 15, 21, 25, ... — each one is a counterexample.
- The claim is refuted by 9 = 3 × 3, the very first composite odd integer.

Try modifying the range to `range(3, 201)`. How does the density of counterexamples
change? What pattern do you notice about which odd numbers are counterexamples?
*(This connects to the Sieve of Eratosthenes — a topic we will revisit in Module 08.)*


### 2 · Invariant Monitoring — Privilege Escalation

**Scenario.** A process privilege level starts at 0 and changes over time through
a series of state transitions (requests, grants, releases). The system invariant is:

$$I(s) : \text{privilege\_level}(s) \leq \texttt{MAX\_PRIVILEGE}$$

We simulate two runs:

- **Safe run** — every transition respects the bound; the invariant is preserved.
- **Vulnerable run** — at step 18, a bug allows a transition that bypasses the check,
  causing privilege to exceed `MAX_PRIVILEGE`. The invariant is violated.

The plot shows privilege level over time; red segments mark invariant violations.


In [ ]:
# ── 2 · Invariant Monitoring ──────────────────────────────────────────────────
import random
random.seed(42)

MAX_PRIVILEGE = 5
NUM_STEPS     = 30

def simulate(introduce_bug=False, bug_step=18):
    """Simulate privilege level over NUM_STEPS transitions.

    Returns (levels, invariant_ok) where invariant_ok[i] = True iff I(s_i) holds.
    """
    levels = [0]
    inv_ok = [True]
    for step in range(1, NUM_STEPS + 1):
        prev  = levels[-1]
        delta = random.choice([-1, 0, 1])
        new   = max(0, prev + delta)
        if introduce_bug and step == bug_step:
            new = MAX_PRIVILEGE + random.randint(1, 3)
        else:
            new = min(new, MAX_PRIVILEGE)
        levels.append(new)
        inv_ok.append(new <= MAX_PRIVILEGE)
    return levels, inv_ok

levels_safe, inv_safe = simulate(introduce_bug=False)
levels_vuln, inv_vuln = simulate(introduce_bug=True, bug_step=18)

steps = list(range(NUM_STEPS + 1))

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
bound_kw = dict(color=TS_GREEN, linestyle='--', lw=1.8,
                label=f'Invariant bound — max = {MAX_PRIVILEGE}')

# Top: safe run
ax1.step(steps, levels_safe, color=TS_BLUE, where='post', lw=2, label='Privilege level')
ax1.axhline(MAX_PRIVILEGE, **bound_kw)
ax1.fill_between(steps, 0, MAX_PRIVILEGE, alpha=0.07, color=TS_GREEN, step='post')
ax1.set_ylim(-0.5, MAX_PRIVILEGE + 4)
ax1.set_ylabel('Privilege Level')
ax1.set_title('Safe Run — invariant $I(s)$ preserved at every step')
ax1.legend(loc='upper right', fontsize=9)
ax1.text(0.02, 0.88, '\u2713  Invariant holds for all steps',
         transform=ax1.transAxes, color=TS_GREEN, fontsize=10, fontweight='bold')

# Bottom: vulnerable run — colour segments by invariant status
for i in range(1, len(steps)):
    ok    = inv_vuln[i]
    color = TS_BLUE if ok else TS_RED
    ax2.step([steps[i-1], steps[i]], [levels_vuln[i-1], levels_vuln[i]],
             color=color, where='post', lw=2)

ax2.axhline(MAX_PRIVILEGE, **bound_kw)
ax2.fill_between(steps, 0, MAX_PRIVILEGE, alpha=0.07, color=TS_GREEN, step='post')

violations = [(i, levels_vuln[i]) for i in range(len(steps)) if not inv_vuln[i]]
for step_i, lvl in violations:
    ax2.scatter(step_i, lvl, color=TS_RED, s=130, zorder=5, edgecolors='white', lw=1)
    ax2.annotate(
        f'Invariant violated!\nlevel = {lvl} > {MAX_PRIVILEGE}',
        xy=(step_i, lvl), xytext=(step_i + 2, lvl + 1),
        fontsize=9, color=TS_RED,
        arrowprops=dict(arrowstyle='->', color=TS_RED, lw=1.2)
    )

ax2.set_ylim(-0.5, MAX_PRIVILEGE + 4)
ax2.set_xlabel('Simulation Step')
ax2.set_ylabel('Privilege Level')
ax2.set_title('Vulnerable Run — privilege escapes the invariant bound at step 18')
ax2.legend(loc='upper right', fontsize=9)
ax2.text(0.02, 0.88, '\u2717  Invariant violated at step 18',
         transform=ax2.transAxes, color=TS_RED, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


**What do you see?**

- The top plot stays entirely below the dashed green bound — the invariant
  $\text{privilege} \leq 5$ is preserved at every step.
- In the bottom plot, the trajectory turns red at step 18 when the bug fires and
  privilege spikes above the bound.

**Questions to consider:**

1. In the safe run, could privilege *accidentally* reach `MAX_PRIVILEGE + 1` if we ran
   for 10,000 steps instead of 30? Why or why not? *(Think about the invariant proof
   structure: initial state holds, $\Rightarrow$ preserved by every step.)*
2. Change `bug_step` to 1 (the very first step). What happens to the rest of the plot?
3. This simulation models one kind of privilege escalation vulnerability. What real-world
   bug class does bypassing a cap check remind you of? *(Hint: integer overflow.)*


### 3 · Proof Strategy Visualizer

The three proof strategies differ in *what they assume* and *what they derive*.
The diagram below maps out the logical flow of each one side by side, using the
goal $P \Rightarrow Q$ as a common reference point.


In [ ]:
# ── 3 · Proof Strategy Comparison ────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(13, 6.5))
fig.patch.set_facecolor('white')

def draw_flowchart(ax, title, subtitle, boxes, box_colors, arrows):
    ax.set_xlim(0, 6)
    ax.set_ylim(-0.5, len(boxes) * 2.2 + 0.5)
    ax.axis('off')
    ax.set_facecolor('white')

    ax.text(3, len(boxes) * 2.2 + 0.2, title,
            ha='center', va='top', fontsize=11, fontweight='bold', color=TS_BLUE)
    ax.text(3, len(boxes) * 2.2 - 0.35, subtitle,
            ha='center', va='top', fontsize=9, color=TS_GRAY, style='italic')

    y_positions = []
    for i, (label, fc) in enumerate(zip(boxes, box_colors)):
        y = (len(boxes) - 1 - i) * 2.0
        y_positions.append(y)
        rect = mpatches.FancyBboxPatch(
            (0.4, y - 0.45), 5.2, 0.9,
            boxstyle='round,pad=0.15',
            facecolor=fc, edgecolor=TS_GRAY, linewidth=1.2
        )
        ax.add_patch(rect)
        ax.text(3, y, label, ha='center', va='center', fontsize=9, color='#1a1a2e')

    for (src, dst, lbl) in arrows:
        y_start = y_positions[src] - 0.45
        y_end   = y_positions[dst] + 0.45
        ax.annotate('', xy=(3, y_end), xytext=(3, y_start),
                    arrowprops=dict(arrowstyle='->', color=TS_GRAY, lw=1.6))
        if lbl:
            ax.text(3.55, (y_start + y_end) / 2, lbl,
                    fontsize=8, color=TS_GRAY, va='center')

draw_flowchart(
    axes[0],
    'Direct Proof',
    'Goal: $P \\Rightarrow Q$',
    ['Assume $P$',
     'Apply definitions,\ntheorems, algebra',
     'Arrive at $Q$',
     '$P \\Rightarrow Q$ $\\;\\square$'],
    [TS_LIGHT, TS_LIGHT, TS_LIGHT, '#d4edda'],
    [(0,1,''), (1,2,''), (2,3,'')]
)

draw_flowchart(
    axes[1],
    'Contrapositive',
    'Goal: $\\neg Q \\Rightarrow \\neg P$',
    ['Assume $\\neg Q$',
     'Reason backward\nfrom failure',
     'Derive $\\neg P$',
     '$P \\Rightarrow Q$ $\\;\\square$'],
    [TS_LIGHT, TS_LIGHT, TS_LIGHT, '#d4edda'],
    [(0,1,''), (1,2,''), (2,3,'equiv.')]
)

draw_flowchart(
    axes[2],
    'Contradiction',
    'Goal: prove $P$ by absurdity',
    ['Assume $\\neg P$',
     'Derive consequences',
     '$Q$ and $\\neg Q$\nsimultaneously',
     '$P$ must be true $\\;\\square$'],
    [TS_LIGHT, TS_LIGHT, '#fde8e8', '#d4edda'],
    [(0,1,''), (1,2,''), (2,3,'absurd')]
)

plt.suptitle('Three Proof Strategies — Same Goal, Different Paths',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**What do you see?**

All three strategies prove $P \Rightarrow Q$ *(if P then Q)* — but they start from
different assumptions and travel different logical paths:

- **Direct**: start with truth ($P$), build toward the target ($Q$).
- **Contrapositive**: start with the failure condition ($\neg Q$ — *not Q*), show the
  premise must also fail ($\neg P$ — *not P*). Equivalent to direct but often cleaner
  when the failure condition is more concrete than the success condition.
- **Contradiction**: assume the opposite of what you want to prove ($\neg P$), derive an
  impossible state (red box — $\Rightarrow\!\Leftarrow$). The impossibility forces $P$ to be true.

In security, **contradiction** is the technique of choice when you want to show that a
bad state is *unreachable* — assume it is reachable, derive consequences, and show they
contradict a known property of the system.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. COUNTEREXAMPLE SEARCH
#    The claim "every number of the form 6k + 1 (k >= 1) is prime" is also false.
#    Modify the counterexample search above to test this claim on k in [1, 50].
#    What is the smallest counterexample? State the refutation formally using
#    the ∀ / ∃ structure from Section 2.
#
# 2. INVARIANT PROOF SKETCH
#    The safe simulation relies on:  new = min(new, MAX_PRIVILEGE)
#    Write an informal proof (in comments) that this single line guarantees the
#    invariant is preserved at every step, assuming it holds at step 0.
#    Structure your argument as: assume I(s_i) holds → show I(s_{i+1}) holds.
#
# 3. STRONGER INVARIANT
#    Add a second invariant: privilege never decreases below 0.
#    Extend the simulation to track violations of BOTH bounds separately,
#    and update the plot to use a lower dashed bound line as well.

# Your work here:


---

## Summary

| Technique | When to use | Security application |
|---|---|---|
| **Direct proof** | $P$ is concrete; steps to $Q$ are natural | Proving role hierarchy properties |
| **Proof by contrapositive** | $\neg Q$ *(not Q)* is easier to work with | Monitoring: symptom → structural cause |
| **Proof by contradiction** | Want to show something is *impossible* | Proving a bad state is unreachable |
| **Counterexample** | Disproving a $\forall$ *(for all)* claim | CVEs; fuzz testing; pen testing |
| **Invariant** | Reasoning about all reachable states | Protocol safety; memory safety; RBAC |

**Core asymmetry to keep front of mind:**

$$\text{Prove } \forall x,\ P(x) : \text{ reason about ALL cases.}$$
$$\text{Disprove } \forall x,\ P(x) : \text{ find ONE counterexample.}$$

---

## Up Next

**Module 01 · Unit 01 — Propositions and Logical Connectives**

We move from informal proof to the formal language of propositional logic. You will
learn to express access control policies as Boolean formulas, evaluate them with
truth tables, and see exactly where short-circuit evaluation creates security
vulnerabilities.

$\rightarrow$ `modules/module-01/unit-01-propositions-connectives.ipynb`
